# Explore kurry transcripts

Quick exploration of the kurry/sp500_earnings_transcripts dataset.

**Provenance note:** This dataset has undocumented origin. Treat as interim scratchpad only; any publishable analysis must be re-run on WRDS `ciq_transcripts` once Tulane entitlement lands. See `.claude/.../open_investigation_kurry_dataset.md`.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PARQUET = PROJECT_ROOT / 'raw_data' / 'transcripts_kurry' / 'transcripts.parquet'

# Just peek at the schema without loading the full text columns
# (reads only column metadata, not row data)
import pyarrow.parquet as pq
meta = pq.read_metadata(PARQUET)
schema = pq.read_schema(PARQUET)
print(f'rows: {meta.num_rows:,}')
print(f'columns: {schema.names}')
print(schema)

In [ ]:
# Fast path: read only lightweight columns (no transcript text)
meta_cols = ['symbol', 'quarter', 'year', 'date', 'company_name', 'company_id']
meta_df = pd.read_parquet(PARQUET, columns=meta_cols)
print(f'{len(meta_df):,} rows')
meta_df.head(10)

In [ ]:
# Read one firm-quarter with full text — e.g. Apple 2023 Q1
import pyarrow.parquet as pq

# Filter at the parquet level (faster than loading then filtering)
filters = [('symbol', '=', 'AAPL'), ('year', '=', 2023), ('quarter', '=', 1)]
one = pd.read_parquet(PARQUET, filters=filters)
print(one[['symbol', 'year', 'quarter', 'date', 'company_name']])
print()
print(f'transcript length: {len(one.iloc[0]["content"]):,} chars')

In [ ]:
# Look at the first few speaker turns from that transcript
structured = one.iloc[0]['structured_content']
for turn in structured[:6]:
    speaker = turn.get('speaker', 'UNKNOWN')
    text = turn.get('text', '')
    print(f'--- {speaker} ---')
    print(text[:400] + ('...' if len(text) > 400 else ''))
    print()

In [ ]:
# Or load an entire firm's history into a DataFrame
aapl = pd.read_parquet(PARQUET, filters=[('symbol', '=', 'AAPL')])
print(f'{len(aapl)} transcripts for AAPL')
aapl[['symbol', 'year', 'quarter', 'date']].sort_values('date')